# HW12 Part 1 — LLM Optimization with GPTQ (5 pts)

**SJSU ID:** `019129357`

Run this notebook **on Google Colab** (GPU runtime). This part only needs ~5-10 minutes on a T4.

### How to run
1. Upload to [Google Colab](https://colab.research.google.com/).
2. **Runtime -> Change runtime type -> GPU** (T4 is fine).
3. **Runtime -> Run all**.
4. At the end, download `hw12_part1_results.json` (Files panel on the left) — Part 3's analysis notebook reads it to build the final writeup, so copy it next to the local notebook (e.g. `C:\Users\anushka\Downloads\tokens\hw12\hw12_part1_results.json`) when it finishes.

## 0. Personal parameters (from SJSU ID)

In [ ]:
SJSU_ID = "019129357"
SID4 = int(SJSU_ID[-4:])  # 9357

SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}  (does NOT add an extra temperature; only used in Part 2)")

## 0.1 Install dependencies

Restart the runtime **once** if Colab asks you to after installing, then re-run from the personal-parameters cell.

In [ ]:
!pip install -q "transformers>=4.44.0" "datasets>=2.19.0" accelerate evaluate \
                matplotlib tqdm sentencepiece protobuf pandas

# GPTQ stack (Transformers docs: Optimum + GPT-QModel)
!pip install -q "optimum>=1.24.0"
!pip install -q "gptqmodel>=1.4.3" --no-build-isolation

print("Installs done. If Colab asks to restart the runtime, restart once, then continue from section 0.")

In [ ]:
import os
import gc
import time
import json
import random

import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, GPTQConfig

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("No GPU detected — set Runtime > Change runtime type > GPU")

In [ ]:
# Shared measurement helpers

def peak_gpu_memory_gb() -> float:
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def reset_peak_memory():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()


def dir_size_mb(path: str) -> float:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 ** 2)


def latency_stats_ms(times_s):
    arr = np.array(times_s) * 1000.0
    return float(np.percentile(arr, 50)), float(np.percentile(arr, 95))


def free_model(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()

---
# Part 1 — LLM optimization with GPTQ (5 pts)

Quantize a small causal LLM with GPTQ, calibrate/evaluate on **WikiText-2**, and compare:
- quality (perplexity)
- p50 / p95 inference latency
- peak GPU memory
- on-disk size

Same hardware (this Colab GPU) for base and quantized runs.

In [ ]:
# No specific base LLM was assigned by the instructor at the time this was written;
# facebook/opt-125m is small, causal, and well-supported by GPTQConfig/gptqmodel.
# Swap this ID if your instructor specifies a different required model.
BASE_LLM = "facebook/opt-125m"
GPTQ_BITS = 4
GPTQ_GROUP_SIZE = 128

BASE_SAVE_DIR = "./models/opt125m_fp16"
GPTQ_SAVE_DIR = "./models/opt125m_gptq4"

MAX_LENGTH = 512
EVAL_STRIDE = 256
LATENCY_WARMUP = 5
LATENCY_RUNS = 50
GEN_MAX_NEW_TOKENS = 32

print(f"Base LLM: {BASE_LLM}")
print(f"GPTQ: {GPTQ_BITS}-bit, group_size={GPTQ_GROUP_SIZE}")

### 1.1 Load WikiText-2

In [ ]:
wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print(wikitext)
print("Sample train text:", repr(wikitext["train"][10]["text"][:120]))

In [ ]:
tokenizer_lm = AutoTokenizer.from_pretrained(BASE_LLM)
if tokenizer_lm.pad_token is None:
    tokenizer_lm.pad_token = tokenizer_lm.eos_token


def build_calibration_texts(dataset_split, n_samples=128, start=SLICE, seed=SEED):
    """Non-empty WikiText lines starting near SLICE for GPTQ calibration."""
    texts = [t for t in dataset_split["text"] if t and t.strip()]
    start = start % max(1, len(texts))
    window = texts[start: start + max(n_samples * 4, n_samples)]
    rng = np.random.default_rng(seed)
    if len(window) > n_samples:
        idx = rng.choice(len(window), size=n_samples, replace=False)
        chosen = [window[i] for i in idx]
    else:
        chosen = window[:n_samples]
    return [c.strip() for c in chosen if len(c.strip()) > 40]


calib_texts = build_calibration_texts(wikitext["train"], n_samples=128)
print(f"Calibration texts: {len(calib_texts)}")
print("First calib snippet:", calib_texts[0][:100], "...")

### 1.2 Perplexity + latency helpers

In [ ]:
@torch.no_grad()
def compute_perplexity(model, tokenizer, texts, max_length=MAX_LENGTH, stride=EVAL_STRIDE, max_chars=200_000):
    """Sliding-window perplexity over concatenated WikiText text."""
    model.eval()
    text = "\n\n".join(t for t in texts if t and t.strip())
    text = text[:max_chars]
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(model.device)
    seq_len = input_ids.size(1)

    nlls = []
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_chunk = input_ids[:, begin:end]
        target_chunk = input_chunk.clone()
        target_chunk[:, :-trg_len] = -100

        outputs = model(input_chunk, labels=target_chunk)
        neg_log_likelihood = outputs.loss * trg_len
        nlls.append(neg_log_likelihood)

        prev_end = end
        if end == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / prev_end)
    return float(ppl.item())


@torch.no_grad()
def measure_generation_latency(model, tokenizer, prompt="The history of artificial intelligence begins",
                               warmup=LATENCY_WARMUP, runs=LATENCY_RUNS,
                               max_new_tokens=GEN_MAX_NEW_TOKENS):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    for _ in range(warmup):
        _ = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()

    times = []
    for _ in range(runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    p50, p95 = latency_stats_ms(times)
    return p50, p95, times

### 1.3 Evaluate base (FP16) LLM

In [ ]:
reset_peak_memory()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_LLM,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()

os.makedirs(BASE_SAVE_DIR, exist_ok=True)
base_model.save_pretrained(BASE_SAVE_DIR)
tokenizer_lm.save_pretrained(BASE_SAVE_DIR)
base_disk_mb = dir_size_mb(BASE_SAVE_DIR)

print("Computing base perplexity on WikiText-2 test...")
base_ppl = compute_perplexity(base_model, tokenizer_lm, wikitext["test"]["text"])
print(f"Base perplexity: {base_ppl:.3f}")

print("Measuring base latency...")
base_p50, base_p95, _ = measure_generation_latency(base_model, tokenizer_lm)
base_peak_gb = peak_gpu_memory_gb()

base_results = {
    "config": "Base LLM (OPT-125M FP16)",
    "perplexity": base_ppl,
    "p50_ms": base_p50,
    "p95_ms": base_p95,
    "peak_mem_gb": base_peak_gb,
    "disk_mb": base_disk_mb,
}
print(json.dumps(base_results, indent=2))

free_model(base_model)

### 1.4 Quantize with GPTQ (WikiText-2 calibration)

Uses Hugging Face `GPTQConfig` ([docs](https://huggingface.co/docs/transformers/quantization/gptq)).

In [ ]:
reset_peak_memory()

gptq_config = GPTQConfig(
    bits=GPTQ_BITS,
    group_size=GPTQ_GROUP_SIZE,
    dataset=calib_texts,
    tokenizer=tokenizer_lm,
    desc_act=False,
)

print("Running GPTQ quantization (this can take a few minutes)...")
gptq_model = AutoModelForCausalLM.from_pretrained(
    BASE_LLM,
    quantization_config=gptq_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
gptq_model.eval()

os.makedirs(GPTQ_SAVE_DIR, exist_ok=True)
gptq_model.save_pretrained(GPTQ_SAVE_DIR)
tokenizer_lm.save_pretrained(GPTQ_SAVE_DIR)
gptq_disk_mb = dir_size_mb(GPTQ_SAVE_DIR)

print("Computing GPTQ perplexity on WikiText-2 test...")
gptq_ppl = compute_perplexity(gptq_model, tokenizer_lm, wikitext["test"]["text"])
print(f"GPTQ perplexity: {gptq_ppl:.3f}")

print("Measuring GPTQ latency...")
gptq_p50, gptq_p95, _ = measure_generation_latency(gptq_model, tokenizer_lm)
gptq_peak_gb = peak_gpu_memory_gb()

gptq_results = {
    "config": "GPTQ-quantized LLM (4-bit)",
    "perplexity": gptq_ppl,
    "p50_ms": gptq_p50,
    "p95_ms": gptq_p95,
    "peak_mem_gb": gptq_peak_gb,
    "disk_mb": gptq_disk_mb,
}
print(json.dumps(gptq_results, indent=2))

free_model(gptq_model)

### 1.5 Table HW11.1 — GPTQ comparison

In [ ]:
import pandas as pd

table_hw11_1 = pd.DataFrame([
    {
        "Configuration": base_results["config"],
        "Quality (perplexity)": round(base_results["perplexity"], 3),
        "p50 / p95 latency (ms)": f"{base_results['p50_ms']:.1f} / {base_results['p95_ms']:.1f}",
        "Peak memory (GB)": round(base_results["peak_mem_gb"], 3),
        "Size on disk (MB)": round(base_results["disk_mb"], 1),
    },
    {
        "Configuration": gptq_results["config"],
        "Quality (perplexity)": round(gptq_results["perplexity"], 3),
        "p50 / p95 latency (ms)": f"{gptq_results['p50_ms']:.1f} / {gptq_results['p95_ms']:.1f}",
        "Peak memory (GB)": round(gptq_results["peak_mem_gb"], 3),
        "Size on disk (MB)": round(gptq_results["disk_mb"], 1),
    },
])

print("Table HW11.1 -- GPTQ comparison on the same LLM/task")
display(table_hw11_1)

### 1.6 Save results for the local Part 2/3 notebook

Download `hw12_part1_results.json` from the Colab Files panel and place it next to `hw12_part2_distillation_local.ipynb` on your machine.

In [ ]:
with open("hw12_part1_results.json", "w") as f:
    json.dump({
        "base_llm": BASE_LLM,
        "gptq_bits": GPTQ_BITS,
        "gptq_group_size": GPTQ_GROUP_SIZE,
        "gpu": torch.cuda.get_device_name(0),
        "base_results": base_results,
        "gptq_results": gptq_results,
    }, f, indent=2)

print("Saved hw12_part1_results.json")
try:
    from google.colab import files
    files.download("hw12_part1_results.json")
except ImportError:
    pass